In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Load dataset
data = fetch_california_housing()
X, y = data.data, data.target

# Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.80, test_size=0.20, random_state=123)
print("Train/Test sizes:", X_train.shape, X_test.shape)

# Individual Models
lr  = LinearRegression()
dt  = DecisionTreeRegressor(random_state=42)
knn = KNeighborsRegressor()

lr.fit(X_train, y_train)
dt.fit(X_train, y_train)
knn.fit(X_train, y_train)

# Evaluate all 3
print("R2 - Linear Regression:", round(r2_score(y_test, lr.predict(X_test)), 3))
print("R2 - Decision Tree:    ", round(r2_score(y_test, dt.predict(X_test)), 3))
print("R2 - KNN:              ", round(r2_score(y_test, knn.predict(X_test)), 3))

Train/Test sizes: (16512, 8) (4128, 8)
R2 - Linear Regression: 0.61
R2 - Decision Tree:     0.613
R2 - KNN:               0.163


In [3]:
from sklearn.ensemble import BaggingRegressor

# Initialize and train Bagging Regressor
bag_regressor = BaggingRegressor(random_state=1, n_jobs=-1)
bag_regressor.fit(X_train, y_train)

# Evaluate
print('Training R2 : %.3f' % bag_regressor.score(X_train, y_train))
print('Test R2     : %.3f' % bag_regressor.score(X_test, y_test))

Training R2 : 0.963
Test R2     : 0.793


In [4]:
%%time
from sklearn.model_selection import GridSearchCV

# Define parameters grid
params = {
    'estimator': [None, LinearRegression()], # None means DecisionTree
    'n_estimators': [20, 50, 100],
    'max_samples': [0.5, 1.0],
    'max_features': [0.5, 1.0],
    'bootstrap': [True, False],
    'bootstrap_features': [True, False]
}

# Run GridSearchCV
bagging_regressor_grid = GridSearchCV(
    BaggingRegressor(random_state=1, n_jobs=-1), 
    param_grid=params, 
    cv=3, 
    n_jobs=-1, 
    verbose=1
)

bagging_regressor_grid.fit(X_train, y_train)

# Display Results
print('\nTrain R^2 Score :', round(bagging_regressor_grid.best_estimator_.score(X_train, y_train), 3))
print('Test R^2 Score  :', round(bagging_regressor_grid.best_estimator_.score(X_test, y_test), 3))
print('Best R^2 Score (Grid Search):', round(bagging_regressor_grid.best_score_, 3))
print('Best Parameters :', bagging_regressor_grid.best_params_)

Fitting 3 folds for each of 96 candidates, totalling 288 fits

Train R^2 Score : 0.973
Test R^2 Score  : 0.815
Best R^2 Score (Grid Search): 0.801
Best Parameters : {'bootstrap': True, 'bootstrap_features': True, 'estimator': None, 'max_features': 1.0, 'max_samples': 1.0, 'n_estimators': 100}
CPU times: total: 5 s
Wall time: 2h 10min 3s
